# 第八阶段综合练习参考答案

## 题 1 参考答案

In [ ]:
import torch
import torch.nn as nn

x = torch.linspace(-3, 3, 100).reshape(-1, 1)
y = x ** 2 + 0.5 + 0.1 * torch.randn_like(x)

model = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(500):
    pred = model(x)
    loss = criterion(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"最终损失 {loss.item():.4f}")

## 题 2 参考答案

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)


def train_and_eval(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(20):
        for bx, by in train_loader:
            pred = model(bx)
            loss = criterion(pred, by)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    correct = total = 0
    with torch.no_grad():
        for bx, by in test_loader:
            _, predicted = torch.max(model(bx), dim=1)
            correct += (predicted == by).sum().item()
            total += by.size(0)
    return correct / total


model1 = nn.Sequential(nn.Linear(64, 10))
model2 = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))

print("无隐层准确率：", round(train_and_eval(model1) * 100, 1), "%")
print("MLP准确率：", round(train_and_eval(model2) * 100, 1), "%")

## 题 3 参考答案

In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = torch.tensor(digits.data, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 快速训练一个简单模型
model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
from torch.utils.data import TensorDataset, DataLoader
loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
for epoch in range(20):
    for bx, by in loader:
        loss = criterion(model(bx), by)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

torch.save(model.state_dict(), "my_model.pt")

new_model = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
new_model.load_state_dict(torch.load("my_model.pt"))

with torch.no_grad():
    pred = new_model(X_test[:10])
    _, predicted = torch.max(pred, dim=1)
print("预测：", predicted.tolist())
print("真实：", y_test[:10].tolist())